# Stage 3 — Farm Quality Scorer
Reads the 24-month farm profile produced by Stage 2 and computes a **composite credit quality score (0–100)**.

Five dimensions:
1. **Productivity** (30%) — average and peak yield potential
2. **Consistency** (25%) — income stability (low CV = reliable repayment)
3. **Trend** (20%) — year-over-year improvement
4. **Vegetation Health** (15%) — NDVI/GNDVI greenness and active growing months
5. **Drought Resilience** (10%) — yield floor during dry/heat-stress periods

## Cell 1 — Imports & config

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# ── Scoring bounds — all derived from training dataset statistics ──────────────
#
# Dataset: "Crop Yield Prediction" Kaggle (1,621 observations, 90 fields, 2023)
#
# PRODUCTIVITY  → p5 / p95 of ALL yield observations
#   p5  = 29.01 t/ha → score 0  |  p95 = 54.53 t/ha → score 100
#
# CONSISTENCY   → min / max of per-field CV — inverted (lower CV = higher score)
#   max CV = 0.269 → score 0  |  min CV = 0.028 → score 100
#
# TREND         → yr2/yr1 ratio. ±15% bounds consistent with dataset CV range.
#   0.85 → score 0  |  1.15 → score 100
#
# VEGETATION    → p10 / p90 of NDVI (NDVI > 0 observations)
#   p10 = 0.18 → score 0  |  p90 = 0.66 → score 100
#   Active-vegetation threshold NDVI > 0.30: Myneni et al. (1995), J. Geophys. Res.
#
# RESILIENCE    → p10 / p90 of yield during stress months
#   Stress = rainfall < 5 mm/month OR temperature > 35°C
#   p10 = 28.5 t/ha → score 0  |  p90 = 50.4 t/ha → score 100
#   Heat-stress threshold 35°C: FAO Irrigation & Drainage Paper No. 56 (Allen et al., 1998)
#
# WEIGHTS — expert-designed, following FAO composite indicator methodology (FAO, 2012)

BOUNDS = {
    "productivity": (29.01, 54.53),
    "consistency":  (0.269, 0.028),
    "trend":        (0.85,  1.15),
    "vegetation":   (0.18,  0.66),
    "resilience":   (28.5,  50.4),
}

WEIGHTS = {
    "productivity": 0.30,
    "consistency":  0.25,
    "trend":        0.20,
    "vegetation":   0.15,
    "resilience":   0.10,
}

PROFILE_PATH = '../data/farm_profiles/farm_33.88_-5.55.csv'
LAT, LON     = 33.88, -5.55
print('Config ready.')

## Cell 2 — Load profile & validate

In [ ]:
df = pd.read_csv(PROFILE_PATH)
assert 'yield_pred'    in df.columns, 'Missing yield_pred — run Stage 2 first'
assert 'NDVI'          in df.columns, 'Missing NDVI'
assert 'soil_moisture' in df.columns, 'Missing soil_moisture'
assert len(df) >= 12,                 f'Need at least 12 months, got {len(df)}'

df = df.sort_values(['year', 'month']).reset_index(drop=True)
df['year']  = df['year'].round().astype(int)
df['month'] = df['month'].round().astype(int)

y0, m0 = int(df.iloc[0]['year']),  int(df.iloc[0]['month'])
y1, m1 = int(df.iloc[-1]['year']), int(df.iloc[-1]['month'])

print(f'Loaded {len(df)} months  ({y0}-{m0:02d}  →  {y1}-{m1:02d})')
print(f'yield_pred : {df["yield_pred"].min():.1f} – {df["yield_pred"].max():.1f}  '
      f'mean={df["yield_pred"].mean():.1f} tons/ha')
print(f'NDVI       : {df["NDVI"].min():.3f} – {df["NDVI"].max():.3f}  '
      f'mean={df["NDVI"].mean():.3f}')
print(f'soil_moist : {df["soil_moisture"].min():.1f} – {df["soil_moisture"].max():.1f} %')
df[['year','month','NDVI','temperature','rainfall','soil_moisture','yield_pred']].round(2)

## Cell 3 — Sub-score functions

In [ ]:
def clip_score(raw, low, high):
    if high == low:
        return 50.0
    return float(np.clip((raw - low) / (high - low) * 100, 0, 100))


def score_productivity(df):
    mean_y = df['yield_pred'].mean()
    peak_y = df['yield_pred'].quantile(0.90)
    blended = 0.70 * mean_y + 0.30 * peak_y
    s = clip_score(blended, *BOUNDS['productivity'])
    return s, {'mean_yield': round(mean_y, 2), 'peak_yield_p90': round(peak_y, 2)}


def score_consistency(df):
    cv = df['yield_pred'].std() / (df['yield_pred'].mean() + 1e-9)
    s = clip_score(cv, *BOUNDS['consistency'])  # inverted: lower CV = higher score
    return s, {'cv': round(cv, 4), 'std': round(df['yield_pred'].std(), 2)}


def score_trend(df):
    n = len(df)
    yr1 = df.iloc[:n//2]['yield_pred'].mean()
    yr2 = df.iloc[n//2:]['yield_pred'].mean()
    ratio = yr2 / (yr1 + 1e-9)
    s = clip_score(ratio, *BOUNDS['trend'])
    return s, {'yr1_mean': round(yr1, 2), 'yr2_mean': round(yr2, 2), 'ratio': round(ratio, 4)}


def score_vegetation(df):
    mean_ndvi    = df['NDVI'].mean()
    mean_gndvi   = df['GNDVI'].mean() if 'GNDVI' in df.columns else mean_ndvi
    green_months = int((df['NDVI'] > 0.30).sum())
    # Score on mean NDVI directly — p10/p90 bounds from training dataset
    s = clip_score(mean_ndvi, *BOUNDS['vegetation'])
    return s, {'mean_ndvi': round(mean_ndvi, 4),
               'mean_gndvi': round(mean_gndvi, 4),
               'green_months': green_months}


def score_resilience(df):
    # Stress: rainfall < 5 mm/month OR temperature > 35°C
    # (FAO Irrigation & Drainage Paper No. 56, Allen et al. 1998)
    stress_mask = (df['rainfall'] < 5) | (df['temperature'] > 35)
    stress_df   = df[stress_mask]
    if len(stress_df) == 0:
        return 100.0, {'stress_months': 0, 'floor_yield': None}
    floor_yield = stress_df['yield_pred'].mean()
    s = clip_score(floor_yield, *BOUNDS['resilience'])
    return s, {'stress_months': int(stress_mask.sum()),
               'floor_yield': round(floor_yield, 2),
               'avg_sm_stress': round(stress_df['soil_moisture'].mean(), 2)}


print('Scoring functions ready.')

## Cell 4 — Compute all scores

In [ ]:
s_prod,  d_prod  = score_productivity(df)
s_cons,  d_cons  = score_consistency(df)
s_trend, d_trend = score_trend(df)
s_veg,   d_veg   = score_vegetation(df)
s_res,   d_res   = score_resilience(df)

sub_scores = {
    'productivity': s_prod,
    'consistency':  s_cons,
    'trend':        s_trend,
    'vegetation':   s_veg,
    'resilience':   s_res,
}

quality_score = sum(sub_scores[k] * WEIGHTS[k] for k in sub_scores)

print('─' * 52)
print(f'  FARM QUALITY SCORE   ({LAT}, {LON})')
print('─' * 52)
print(f'  Productivity   ({WEIGHTS["productivity"]*100:.0f}%)  →  {s_prod:5.1f} / 100')
print(f'    mean yield={d_prod["mean_yield"]} t/ha  peak={d_prod["peak_yield_p90"]} t/ha')
print()
print(f'  Consistency    ({WEIGHTS["consistency"]*100:.0f}%)  →  {s_cons:5.1f} / 100')
print(f'    CV={d_cons["cv"]}  std={d_cons["std"]} t/ha')
print()
print(f'  Trend          ({WEIGHTS["trend"]*100:.0f}%)  →  {s_trend:5.1f} / 100')
print(f'    yr1={d_trend["yr1_mean"]}  yr2={d_trend["yr2_mean"]}  ratio={d_trend["ratio"]}')
print()
print(f'  Vegetation     ({WEIGHTS["vegetation"]*100:.0f}%)  →  {s_veg:5.1f} / 100')
print(f'    NDVI={d_veg["mean_ndvi"]}  GNDVI={d_veg["mean_gndvi"]}  green months={d_veg["green_months"]}/24')
print()
print(f'  Resilience     ({WEIGHTS["resilience"]*100:.0f}%)  →  {s_res:5.1f} / 100')
if d_res['stress_months'] > 0:
    print(f'    stress months={d_res["stress_months"]}  floor yield={d_res["floor_yield"]} t/ha  sm={d_res["avg_sm_stress"]}%')
else:
    print(f'    no stress periods detected')
print()
print('─' * 52)
print(f'  COMPOSITE SCORE  →  {quality_score:.1f} / 100')
print('─' * 52)

## Cell 5 — Radar chart + yield timeline

In [ ]:
SCORE_COLOR = '#3498db'  # fixed blue — no tier coloring

fig = plt.figure(figsize=(16, 6))
fig.patch.set_facecolor('#f8f9fa')

# ── LEFT: Radar chart ──────────────────────────────────────────
ax_radar = fig.add_subplot(131, polar=True)
labels  = ['Productivity', 'Consistency', 'Trend', 'Vegetation', 'Resilience']
values  = [sub_scores[k] for k in ['productivity','consistency','trend','vegetation','resilience']]
N       = len(labels)
angles  = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
vals    = values + values[:1]

ax_radar.set_theta_offset(np.pi / 2)
ax_radar.set_theta_direction(-1)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(labels, fontsize=8)
ax_radar.set_ylim(0, 100)
ax_radar.set_yticks([25, 50, 75, 100])
ax_radar.set_yticklabels(['25','50','75','100'], fontsize=6, color='grey')
ax_radar.plot(angles, vals, color=SCORE_COLOR, linewidth=2)
ax_radar.fill(angles, vals, color=SCORE_COLOR, alpha=0.25)
ax_radar.set_title(f'Quality Dimensions\nScore: {quality_score:.1f} / 100',
                    fontsize=10, fontweight='bold', pad=15)

# ── MIDDLE: Yield timeline ─────────────────────────────────────
ax_yield = fig.add_subplot(132)
ax_yield.set_facecolor('#f8f9fa')
x        = np.arange(len(df))
labels_x = [f"{int(r['year'])}-{int(r['month']):02d}" for _, r in df.iterrows()]
half     = len(df) // 2

ax_yield.fill_between(x[:half+1], df['yield_pred'].values[:half+1], alpha=0.15, color='steelblue')
ax_yield.fill_between(x[half-1:], df['yield_pred'].values[half-1:], alpha=0.15, color='darkorange')
ax_yield.plot(x, df['yield_pred'].values, color='steelblue', lw=2, marker='o', ms=3)
ax_yield.axvline(half, color='grey', lw=1, ls='--', alpha=0.5)
ax_yield.axhline(df['yield_pred'].mean(), color='red', lw=1, ls=':', alpha=0.7,
                  label=f"mean={df['yield_pred'].mean():.1f}")

stress_mask = (df['rainfall'] < 5) | (df['temperature'] > 35)
for i, stressed in enumerate(stress_mask):
    if stressed:
        ax_yield.axvspan(i-0.4, i+0.4, color='#e74c3c', alpha=0.12)

ax_yield.set_xticks(x[::3])
ax_yield.set_xticklabels(labels_x[::3], rotation=45, ha='right', fontsize=7)
ax_yield.set_ylabel('Yield (tons/ha)', fontsize=9)
ax_yield.set_title('Predicted Yield Timeline\n(red zones = stress periods)', fontsize=10)
ax_yield.legend(fontsize=8)
ax_yield.text(half/2,      df['yield_pred'].max()*0.97, 'Year 1', ha='center', fontsize=8, color='steelblue')
ax_yield.text(half+half/2, df['yield_pred'].max()*0.97, 'Year 2', ha='center', fontsize=8, color='darkorange')

# ── RIGHT: Score card ──────────────────────────────────────────
ax_card = fig.add_subplot(133)
ax_card.set_facecolor('#f8f9fa')
ax_card.axis('off')

dim_colors  = ['#2ecc71','#3498db','#f39c12','#9b59b6','#1abc9c']
bar_labels  = ['Productivity','Consistency','Trend','Vegetation','Resilience']
bar_values  = [sub_scores[k] for k in ['productivity','consistency','trend','vegetation','resilience']]
y_pos       = [0.72, 0.60, 0.48, 0.36, 0.24]

for lbl, val, yp, col in zip(bar_labels, bar_values, y_pos, dim_colors):
    ax_card.add_patch(FancyBboxPatch((0.02, yp-0.04), 0.95, 0.09,
                      boxstyle='round,pad=0.01', facecolor='#e0e0e0', edgecolor='none',
                      transform=ax_card.transAxes))
    ax_card.add_patch(FancyBboxPatch((0.02, yp-0.04), 0.95 * val/100, 0.09,
                      boxstyle='round,pad=0.01', facecolor=col, alpha=0.85, edgecolor='none',
                      transform=ax_card.transAxes))
    ax_card.text(0.04, yp+0.005, lbl, transform=ax_card.transAxes,
                  fontsize=8, color='white', fontweight='bold', va='center')
    ax_card.text(0.96, yp+0.005, f'{val:.0f}', transform=ax_card.transAxes,
                  fontsize=9, color='#333', fontweight='bold', va='center', ha='right')

ax_card.add_patch(FancyBboxPatch((0.15, 0.83), 0.70, 0.14,
                  boxstyle='round,pad=0.02', facecolor=SCORE_COLOR, edgecolor='none',
                  transform=ax_card.transAxes, alpha=0.9))
ax_card.text(0.50, 0.90, f'{quality_score:.1f} / 100',
              transform=ax_card.transAxes, fontsize=20, color='white',
              fontweight='bold', ha='center', va='center')
ax_card.set_title('Credit Score Card', fontsize=10)

plt.suptitle(f'Farm Credit Assessment  |  GPS ({LAT}, {LON})  |  Meknès, Morocco',
              fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'../data/farm_profiles/score_card_{LAT}_{LON}.png',
             dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Score card saved.')

## Cell 6 — NDVI & climate context

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
fig.patch.set_facecolor('#f8f9fa')
fig.suptitle(f'Farm Climate & Vegetation Profile  ({LAT}, {LON})', fontsize=11, fontweight='bold')

x = np.arange(len(df))
xlbl = [f"{int(r['year'])}-{int(r['month']):02d}" for _, r in df.iterrows()]

def style_ax(ax, title, ylabel, color):
    ax.set_facecolor('#f8f9fa')
    ax.set_title(title, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks(x[::3])
    ax.set_xticklabels(xlbl[::3], rotation=45, ha='right', fontsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.spines[['top','right']].set_visible(False)

# NDVI
axes[0,0].fill_between(x, df['NDVI'].values, alpha=0.2, color='green')
axes[0,0].plot(x, df['NDVI'].values, color='green', lw=2)
axes[0,0].axhline(0.30, color='gray', ls='--', lw=1, label='active vegetation (0.30)')
axes[0,0].legend(fontsize=7)
style_ax(axes[0,0], 'NDVI (vegetation greenness)', 'NDVI', 'green')

# Soil moisture
axes[0,1].fill_between(x, df['soil_moisture'].values, alpha=0.2, color='saddlebrown')
axes[0,1].plot(x, df['soil_moisture'].values, color='saddlebrown', lw=2)
style_ax(axes[0,1], 'Soil Moisture (%)', 'sm (%)', 'saddlebrown')

# Rainfall
axes[1,0].bar(x, df['rainfall'].values, color='royalblue', alpha=0.7)
style_ax(axes[1,0], 'Monthly Rainfall (mm)', 'mm', 'royalblue')

# Temperature
axes[1,1].fill_between(x, df['temperature'].values, alpha=0.15, color='tomato')
axes[1,1].plot(x, df['temperature'].values, color='tomato', lw=2)
axes[1,1].axhline(35, color='red', ls='--', lw=1, label='heat stress (35°C)')
axes[1,1].legend(fontsize=7)
style_ax(axes[1,1], 'Temperature (°C)', '°C', 'tomato')

plt.tight_layout()
plt.show()

## Cell 7 — Export JSON report (for WhatsApp / API integration)

In [ ]:
import json
from datetime import date

r0 = df.iloc[0]
r1 = df.iloc[-1]

# NDVI linear trend slope (per month)
ndvi_slope = float(np.polyfit(np.arange(len(df)), df['NDVI'].values, 1)[0])

peak_ndvi_row = df.loc[df['NDVI'].idxmax()]

stress_mask = (df['rainfall'] < 5) | (df['temperature'] > 35)
stress_months_list = [
    f"{int(r['year'])}-{int(r['month']):02d}"
    for _, r in df[stress_mask].iterrows()
]

dry_months  = df[df['rainfall'] < 10]
irrig_proxy = (round(float(dry_months['soil_moisture'].mean()), 2)
               if len(dry_months) > 0 else None)

report = {
    'farm': {'lat': LAT, 'lon': LON, 'region': 'Meknès, Morocco'},
    'assessment_date': str(date.today()),
    'data_window': {
        'from':   f"{int(r0['year'])}-{int(r0['month']):02d}",
        'to':     f"{int(r1['year'])}-{int(r1['month']):02d}",
        'months': len(df),
    },
    'quality_score': round(quality_score, 1),
    'sub_scores': {k: round(v, 1) for k, v in sub_scores.items()},

    'vegetation': {
        'mean_ndvi':            d_veg['mean_ndvi'],
        'mean_gndvi':           d_veg['mean_gndvi'],
        'peak_ndvi':            round(float(df['NDVI'].max()), 4),
        'peak_ndvi_month':      f"{int(peak_ndvi_row['year'])}-{int(peak_ndvi_row['month']):02d}",
        'green_months':         d_veg['green_months'],
        'ndvi_slope_per_month': round(ndvi_slope, 6),
        'ndvi_trend':           'improving' if ndvi_slope > 0 else 'declining',
    },

    'climate': {
        'avg_temperature_c':     round(float(df['temperature'].mean()), 2),
        'avg_rainfall_mm_month': round(float(df['rainfall'].mean()), 2),
        'total_rainfall_mm':     round(float(df['rainfall'].sum()), 1),
        'avg_soil_moisture_pct': round(float(df['soil_moisture'].mean()), 2),
        'dry_season_sm_pct':     irrig_proxy,
        'stress_months_count':   int(stress_mask.sum()),
        'stress_months':         stress_months_list,
    },

    'resilience': {
        'avg_sm_during_stress': d_res.get('avg_sm_stress'),
        'stress_months_count':  d_res['stress_months'],
    },
}

out_path = f'../data/farm_profiles/report_{LAT}_{LON}.json'
with open(out_path, 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print(f'\nReport saved -> {out_path}')